# SMX Quickstart — Binary Classification with a Synthetic Spectral Dataset

This notebook walks through the complete SMX pipeline:

1. Generate a synthetic two-class spectral dataset (no external files needed)
2. Split calibration / test sets
3. Mean-centre the spectra
4. Train an SVM classifier
5. Run the SMX explanation pipeline
6. Inspect ranked spectral zones
7. Visualise threshold-spectrum overlays inline

In [7]:
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split

from smx import SMX, generate_synthetic_spectral_data
from smx.graph.interpretation import reconstruct_threshold_to_spectrum
import plotly.graph_objects as go

SEED = 42
print("Imports OK")

Imports OK


## 1. Synthetic Spectral Dataset

Two-class XRF-like dataset:
- **Class A** — peaks at 250, 380, 550, 700, 850 (channel units)
- **Class B** — peaks at 50, 250, 380, 550, 850 (overlaps A except for the exclusive zones)

Spectral axis: 500 points from 1 to 1000 (channel numbers)

In [8]:
CLASSES_CONFIG = [
    {
        "name": "A",
        "n_samples": 156,
        "peaks": [250, 380, 550, 700, 850],
        "amplitude_mean": 1.0,
        "amplitude_std": 0.3,
        "width_mean": 15.0,
        "width_std": 2.0,
        "noise_std": 0.04,
    },
    {
        "name": "B",
        "n_samples": 146,
        "peaks": [50, 250, 380, 550, 850],
        "amplitude_mean": 1.4,
        "amplitude_std": 0.5,
        "width_mean": 15.0,
        "width_std": 1.8,
        "noise_std": 0.035,
    },
]

df = generate_synthetic_spectral_data(
    classes_config=CLASSES_CONFIG,
    n_points=500,
    x_min=1,
    x_max=1000,
    seed=0,
)

X = df.drop(columns=["Class"])
y = df["Class"]

print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{y.value_counts().to_string()}")
df.head()

Dataset shape: (302, 501)
Class distribution:
Class
A    156
B    146


,Class,1.0,3.002004008016032,5.004008016032064,7.006012024048097,9.008016032064129,11.01002004008016,13.012024048096194,15.014028056112226,17.016032064128257,...,981.9819639278558,983.9839679358718,985.9859719438879,987.9879759519039,989.9899799599199,991.9919839679359,993.993987975952,995.995991983968,997.997995991984,1000.0
0,A,0.070562,0.016006,0.039150,0.089636,0.074702,-0.039091,0.038004,-0.006054,-0.004129,...,-0.041197,-0.013998,0.044011,0.051921,0.107849,-0.002957,-0.026342,-0.020569,-0.040722,-0.003114
1,A,0.035810,0.054999,-0.053288,-0.078745,-0.026402,0.007033,0.019948,0.041919,0.011371,...,0.022239,0.035699,-0.016893,0.004189,0.009122,0.008059,0.021631,-0.072723,-0.001973,0.009561
2,A,0.040627,0.028042,-0.016699,-0.043900,0.068492,-0.031685,-0.041821,-0.043394,0.044692,...,-0.034210,0.050289,-0.059315,-0.052376,0.032714,0.009528,0.004209,-0.003666,0.001251,-0.003684
3,A,-0.063379,0.033778,-0.048515,0.011351,-0.011288,-0.046328,-0.064774,-0.020442,0.069625,...,-0.076550,0.026542,-0.006163,0.047744,-0.003926,-0.035465,-0.005894,0.042392,0.001050,-0.004573
4,A,-0.048643,0.005627,-0.029747,-0.006360,0.009602,0.004006,-0.019007,0.050918,-0.067845,...,-0.002794,0.010005,-0.040877,-0.046018,-0.033444,0.025688,0.010352,0.041610,-0.007468,-0.045746


## 2. Calibration / Test Split and Mean-Centring Preprocessing

In [9]:
X_cal, X_test, y_cal, y_test = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_cal = X_cal.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_cal = y_cal.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

# Mean-centre (fit on calibration set only)
X_mean = X_cal.mean()
X_cal_prep = X_cal - X_mean
X_test_prep = X_test - X_mean

print(f"Calibration set: {X_cal.shape}  |  Test set: {X_test.shape}")
print(f"Cal class distribution: {y_cal.value_counts().to_dict()}")
print(f"Test class distribution: {y_test.value_counts().to_dict()}")

Calibration set: (211, 500)  |  Test set: (91, 500)
Cal class distribution: {'A': 109, 'B': 102}
Test class distribution: {'A': 47, 'B': 44}


## 3. SVM Classifier

In [10]:
svm = SVC(kernel="rbf", C=1.0, probability=True, random_state=SEED)
svm.fit(X_cal_prep, y_cal)

acc = (svm.predict(X_test_prep) == y_test).mean()
print(f"SVM test accuracy: {acc:.2%}")

# Continuous output used by SMX (probability of class A)
class_order = list(svm.classes_)
class_a_idx = class_order.index("A")
y_pred_cal = pd.Series(svm.predict_proba(X_cal_prep)[:, class_a_idx])
print(f"y_pred_cal range: [{y_pred_cal.min():.3f}, {y_pred_cal.max():.3f}]")

SVM test accuracy: 98.90%
y_pred_cal range: [0.000, 0.999]


## 4. SMX Pipeline

Spectral cuts mirror the zones defined for the synthetic dataset:
- **F1** = exclusive Class B peak region (1 – 100)
- **F2** = shared peak region (200 – 300)
- **F3** = shared peak region (330 – 430)
- **F4** = shared peak region (500 – 600)
- **F5** = exclusive Class A peak region (660 – 750)
- **F6** = shared peak region (815 – 890)

In [11]:
spectral_cuts = [
    ("F1",          1.0,   100.0),
    ("background1", 100.0, 200.0),
    ("F2",          200.0, 300.0),
    ("background2", 300.0, 330.0),
    ("F3",          330.0, 430.0),
    ("background3", 430.0, 500.0),
    ("F4",          500.0, 600.0),
    ("background4", 600.0, 660.0),
    ("F5",          660.0, 750.0),
    ("background5", 750.0, 815.0),
    ("F6",          815.0, 890.0),
    ("background6", 890.0, 1000.0),
]

smx = SMX(
    spectral_cuts=spectral_cuts,
    quantiles=[0.25, 0.50, 0.75],
    seeds=[0, 1, 2, 3],
    n_bags=10,
    n_samples_fraction=0.8,
    min_samples_fraction=0.2,
    metric="perturbation",
    estimator=svm,
    perturbation_mode="median",
    perturbation_metric="probability_shift",
)

print("Fitting SMX pipeline…")
smx.fit(X_cal_prep, y_pred_cal, X_cal_natural=X_cal)
print("Done.")

Fitting SMX pipeline…


Bag_1 | samples: yes | predicates: no (all) | valid: 65 | discarded: 7
Bag_2 | samples: yes | predicates: no (all) | valid: 62 | discarded: 10
Bag_3 | samples: yes | predicates: no (all) | valid: 64 | discarded: 8
Bag_4 | samples: yes | predicates: no (all) | valid: 62 | discarded: 10
Bag_5 | samples: yes | predicates: no (all) | valid: 62 | discarded: 10
Bag_6 | samples: yes | predicates: no (all) | valid: 61 | discarded: 11
Bag_7 | samples: yes | predicates: no (all) | valid: 61 | discarded: 11
Bag_8 | samples: yes | predicates: no (all) | valid: 67 | discarded: 5
Bag_9 | samples: yes | predicates: no (all) | valid: 65 | discarded: 7
Bag_10 | samples: yes | predicates: no (all) | valid: 61 | discarded: 11

Total bidirectional pairs found: 66

CONSTRUCTED GRAPH SUMMARY
Edges (after removing 66 bidirectional): 263
Predicate nodes: 71
Metric: Perturbation
Variance-exp weighting: ENABLED

Processing graph LRC…
Bag_1 | samples: yes | predicates: no (all) | valid: 62 | discarded: 10
Bag_2 

## 5. Results — Top Predicates by Local Reaching Centrality

In [12]:
top = (
    smx.lrc_natural_[smx.lrc_natural_["Zone"].notna()]
    .sort_values("Local_Reaching_Centrality", ascending=False)
    [["Zone", "Operator", "Threshold_Natural", "Local_Reaching_Centrality"]]
    .reset_index(drop=True)
)

print("Top predicates by Local Reaching Centrality:")
display(top)

Top predicates by Local Reaching Centrality:


,Zone,Operator,Threshold_Natural,Local_Reaching_Centrality
0,F1,>,2.421081,16.100173
1,F1,>,-2.426950,12.162427
2,F1,>,-2.510487,6.633638
3,F1,<=,2.421081,2.414581
4,F5,>,1.830423,1.688905
...,...,...,...,...
67,background4,<=,-0.039792,0.000139
68,background3,<=,-0.000162,0.000138
69,background3,<=,-0.029470,0.000138
70,background6,<=,0.000471,0.000109


## 6. Threshold-Spectrum Visualisations

For each zone, the top-ranked predicate threshold is reconstructed back into
spectrum space (red dashed line) and overlaid on the calibration spectra coloured by class.

In [14]:
CLASS_COLORS = {"A": "gold", "B": "blue"}

def plot_zone_inline(lrc_natural_df, row_index, spectral_zones_original,
                     pca_info_dict_original, y_labels, class_colors=None):
    """Reconstruct a threshold to spectrum space and display the plot inline."""
    if class_colors is None:
        class_colors = {"A": "gold", "B": "blue"}

    zone_name = lrc_natural_df.iloc[row_index]["Zone"]
    threshold_score = float(lrc_natural_df.iloc[row_index]["Threshold_Natural"])

    threshold_spectrum = reconstruct_threshold_to_spectrum(
        threshold_value=threshold_score,
        zone_name=zone_name,
        pca_info_dict=pca_info_dict_original,
    )

    zone_df = spectral_zones_original[zone_name]
    x_values = pd.to_numeric(zone_df.columns, errors="coerce")

    fig = go.Figure()
    seen_classes = set()
    for idx, row in zone_df.iterrows():
        class_label = y_labels.iloc[idx] if idx < len(y_labels) else "Unknown"
        show_legend = class_label not in seen_classes
        seen_classes.add(class_label)
        fig.add_trace(
            go.Scatter(
                x=x_values,
                y=row.values,
                mode="lines",
                line=dict(color=class_colors.get(class_label, "rgba(128,128,128,0.3)"), width=0.5),
                name=f"Class {class_label}",
                legendgroup=class_label,
                showlegend=show_legend,
                hoverinfo="skip",
            )
        )

    node_natural = lrc_natural_df.iloc[row_index].get("Node_Natural", "")
    fig.add_trace(
        go.Scatter(
            x=x_values,
            y=threshold_spectrum.values,
            mode="lines",
            line=dict(color="red", width=4, dash="dash"),
            name=f"Threshold Spectrum ({threshold_spectrum.name})",
        )
    )

    fig.update_layout(
        title=f"Zone '{zone_name}' — Multivariate Threshold (Predicate: {node_natural})",
        xaxis_title="Variables (Artificial Units)",
        yaxis_title="Intensity (Arbitrary Units)",
        template="plotly_white",
        showlegend=True,
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),
    )
    fig.show()


top_per_zone = (
    smx.lrc_natural_[smx.lrc_natural_["Zone"].notna()]
    .sort_values("Local_Reaching_Centrality", ascending=False)[:10]
)

for _, row in top_per_zone.iterrows():
    zone_name = row["Zone"]
    row_index = smx.lrc_natural_.index[
        smx.lrc_natural_["Node"] == row["Node"]
    ].tolist()[0]
    plot_zone_inline(
        lrc_natural_df=smx.lrc_natural_,
        row_index=row_index,
        spectral_zones_original=smx.zones_natural_,
        pca_info_dict_original=smx.pca_info_natural_,
        y_labels=y_cal,
        class_colors=CLASS_COLORS,
    )